# 기본 베이스 라인

In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

# 파일 경로 설정부
file_path = r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing\260509_view_delete\Membership_v2.csv"

# 사용 컬럼 설정부
use_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "is_repurchase",
]

# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)

# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False
        )

# 데이터 로드부
df = pd.read_csv(file_path, usecols=use_cols).copy()

# 숫자형 변환부
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[
    [
        "price",
        "max_screen",
        "is_promotion",
        "is_churn_prevented",
        "payment_device",
        "is_user_verified",
        "gender",
        "age",
    ]
].copy()

# 양성 클래스 정의부
# is_repurchase == 0 을 예측 목표로 두기 때문에 0이면 1, 1이면 0으로 변환
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
numeric_features = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "age",
]

categorical_features = [
    "payment_device",
    "gender",
]

# 전처리 파이프라인 구성부
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 모델 정의부
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
    ),
    "SVM": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=42,
    ),
}

# 평가 수행부
results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]

    result = {
        "model": model_name,
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
    }

    results.append(result)

# 결과 출력부
results_df = (
    pd.DataFrame(results)
    .set_index("model")
    [["precision", "recall", "f1_score", "roc_auc", "pr_auc"]]
    .round(4)
    .sort_values("f1_score", ascending=False)
)

print("양성 클래스 기준: is_repurchase == 0")
print(results_df)

양성 클래스 기준: is_repurchase == 0
                    precision  recall  f1_score  roc_auc  pr_auc
model                                                           
SVM                    0.3450  0.5776    0.4320   0.5772  0.3389
LogisticRegression     0.3379  0.5768    0.4261   0.5800  0.3393
RandomForest           0.3337  0.5196    0.4064   0.5615  0.3290
GradientBoosting       0.3333  0.0008    0.0015   0.5873  0.3460


In [3]:
import warnings
import pandas as pd

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# 실행 경고 정리부
warnings.filterwarnings(
    "ignore",
    message=(
        "X does not have valid feature names, but "
        "LGBMClassifier was fitted with feature names"
    ),
)

# 출력 형식 정리부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.expand_frame_repr", False)

# 파일 경로 설정부
file_path = (
    r"C:\myCode\ott-churn-prediction\kim.kwangil\preprocessing"
    r"\260509_view_delete\Membership_v2.csv"
)

# 사용 컬럼 설정부
use_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "is_repurchase",
]

# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)

# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

# 점수 계산 함수부
def evaluate_scores(y_true, y_pred, y_proba):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }

# 데이터 로드부
df = pd.read_csv(file_path, usecols=use_cols).copy()

# 숫자형 변환부
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df["is_repurchase"])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[
    [
        "price",
        "max_screen",
        "is_promotion",
        "is_churn_prevented",
        "payment_device",
        "is_user_verified",
        "gender",
        "age",
    ]
].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 정의부
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
numeric_features = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
    "age",
]

categorical_features = [
    "payment_device",
    "gender",
]

# 전처리 파이프라인 구성부
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 양성 클래스 가중치 계산부
positive_count = int(y_train.sum())
negative_count = int(len(y_train) - positive_count)
scale_pos_weight = negative_count / positive_count

# 모델 정의부
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
    ),
    "SVM": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=42,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        class_weights=[1.0, scale_pos_weight],
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
    ),
}

# 평가 수행부
results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    clf.fit(X_train, y_train)

    # 학습 데이터 예측부
    y_train_pred = clf.predict(X_train)
    y_train_proba = clf.predict_proba(X_train)[:, 1]

    # 테스트 데이터 예측부
    y_test_pred = clf.predict(X_test)
    y_test_proba = clf.predict_proba(X_test)[:, 1]

    # 학습 데이터 점수 계산부
    train_scores = evaluate_scores(y_train, y_train_pred, y_train_proba)

    # 테스트 데이터 점수 계산부
    test_scores = evaluate_scores(y_test, y_test_pred, y_test_proba)

    # ROC AUC gap 계산부
    roc_auc_gap = train_scores["roc_auc"] - test_scores["roc_auc"]

    # 과적합 여부 판단부
    overfit_flag = "과적합 의심" if roc_auc_gap >= 0.03 else "과적합 아님"

    # 결과 저장부
    result = {
        "model": model_name,
        "precision": test_scores["precision"],
        "recall": test_scores["recall"],
        "f1": test_scores["f1"],
        "train_roc_auc": train_scores["roc_auc"],
        "test_roc_auc": test_scores["roc_auc"],
        "overfit": overfit_flag,
    }

    results.append(result)

# 결과 출력부
results_df = (
    pd.DataFrame(results)
    .set_index("model")
    [
        [
            "precision",
            "recall",
            "f1",
            "train_roc_auc",
            "test_roc_auc",
            "overfit",
        ]
    ]
    .round(
        {
            "precision": 4,
            "recall": 4,
            "f1": 4,
            "train_roc_auc": 4,
            "test_roc_auc": 4,
        }
    )
    .sort_values(["f1", "test_roc_auc"], ascending=False)
)

print("양성 클래스 기준: is_repurchase == 0")
print("precision, recall, f1 는 test 기준")
print("train_roc_auc, test_roc_auc 는 roc_auc_score 기준")
print("과적합 판단 기준: train_roc_auc - test_roc_auc >= 0.03")
print(results_df.to_string())


양성 클래스 기준: is_repurchase == 0
precision, recall, f1 는 test 기준
train_roc_auc, test_roc_auc 는 roc_auc_score 기준
과적합 판단 기준: train_roc_auc - test_roc_auc >= 0.03
                    precision  recall      f1  train_roc_auc  test_roc_auc overfit
model                                                                             
SVM                    0.3450  0.5776  0.4320         0.5846        0.5772  과적합 아님
LogisticRegression     0.3379  0.5768  0.4261         0.5711        0.5800  과적합 아님
LightGBM               0.3426  0.5580  0.4245         0.6304        0.5784  과적합 의심
CatBoost               0.3367  0.5520  0.4183         0.6195        0.5817  과적합 의심
XGBoost                0.3385  0.5452  0.4177         0.6127        0.5823  과적합 의심
RandomForest           0.3337  0.5196  0.4064         0.6495        0.5615  과적합 의심
GradientBoosting       0.3333  0.0008  0.0015         0.5929        0.5873  과적합 아님


# 파생변수 추가

In [4]:
import warnings
import pandas as pd

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# 실행 경고 정리부
warnings.filterwarnings(
    "ignore",
    message=(
        "X does not have valid feature names, but "
        "LGBMClassifier was fitted with feature names"
    ),
)

# 출력 형식 정리부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.expand_frame_repr", False)

# 파일 경로 설정부
file_path = (
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable"
    r"\260519_derived_membership_age_specific.csv"
)

# 원본 전체 컬럼 설정부
raw_source_cols = [
    "USER_KEY",
    "product_code",
    "price",
    "billing_method",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "reg_date",
    "reg_hour",
    "end_date",
    "is_repurchase",
]

# 원본 사용 컬럼 설정부
base_feature_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 타깃 컬럼 설정부
target_col = "is_repurchase"

# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)

# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

# 점수 계산 함수부
def evaluate_scores(y_true, y_pred, y_proba):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }

# 입력 컬럼 추출 함수부
def get_feature_columns(df):
    missing_cols = [
        col for col in base_feature_cols + [target_col] if col not in df.columns
    ]
    if missing_cols:
        raise ValueError(f"필수 컬럼 누락: {missing_cols}")

    derived_feature_cols = [
        col for col in df.columns if col not in raw_source_cols
    ]

    feature_cols = base_feature_cols + derived_feature_cols

    return feature_cols

# 숫자형, 범주형 컬럼 분리 함수부
def split_feature_types(X):
    numeric_features = X.select_dtypes(include="number").columns.tolist()
    categorical_features = [
        col for col in X.columns if col not in numeric_features
    ]
    return numeric_features, categorical_features

# 데이터 로드부
df = pd.read_csv(file_path).copy()

# 사용 컬럼 추출부
feature_cols = get_feature_columns(df)

# 숫자형 변환부
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df[target_col])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 정의부
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
numeric_features, categorical_features = split_feature_types(X)

# 전처리 파이프라인 구성부
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# 학습/평가 데이터 분리부
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 양성 클래스 가중치 계산부
positive_count = int(y_train.sum())
negative_count = int(len(y_train) - positive_count)
scale_pos_weight = negative_count / positive_count

# 모델 정의부
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        random_state=42,
    ),
    "SVM": SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        probability=True,
        class_weight="balanced",
        random_state=42,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
        scale_pos_weight=scale_pos_weight,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        learning_rate=0.05,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        class_weights=[1.0, scale_pos_weight],
        random_seed=42,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
    ),
}

# 평가 수행부
results = []

for model_name, model in models.items():
    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    clf.fit(X_train, y_train)

    # 학습 데이터 예측부
    y_train_pred = clf.predict(X_train)
    y_train_proba = clf.predict_proba(X_train)[:, 1]

    # 테스트 데이터 예측부
    y_test_pred = clf.predict(X_test)
    y_test_proba = clf.predict_proba(X_test)[:, 1]

    # 학습 데이터 점수 계산부
    train_scores = evaluate_scores(y_train, y_train_pred, y_train_proba)

    # 테스트 데이터 점수 계산부
    test_scores = evaluate_scores(y_test, y_test_pred, y_test_proba)

    # ROC AUC gap 계산부
    roc_auc_gap = train_scores["roc_auc"] - test_scores["roc_auc"]

    # 과적합 여부 판단부
    overfit_flag = "과적합 의심" if roc_auc_gap >= 0.03 else "과적합 아님"

    # 결과 저장부
    result = {
        "model": model_name,
        "precision": test_scores["precision"],
        "recall": test_scores["recall"],
        "f1": test_scores["f1"],
        "train_roc_auc": train_scores["roc_auc"],
        "test_roc_auc": test_scores["roc_auc"],
        "overfit": overfit_flag,
    }

    results.append(result)

# 결과 출력부
results_df = (
    pd.DataFrame(results)
    .set_index("model")
    [
        [
            "precision",
            "recall",
            "f1",
            "train_roc_auc",
            "test_roc_auc",
            "overfit",
        ]
    ]
    .round(
        {
            "precision": 4,
            "recall": 4,
            "f1": 4,
            "train_roc_auc": 4,
            "test_roc_auc": 4,
        }
    )
    .sort_values(["f1", "test_roc_auc"], ascending=False)
)

print(f"사용한 입력 컬럼 수: {len(feature_cols)}")
print("양성 클래스 기준: is_repurchase == 0")
print("precision, recall, f1 는 test 기준")
print("train_roc_auc, test_roc_auc 는 roc_auc_score 기준")
print("과적합 판단 기준: train_roc_auc - test_roc_auc >= 0.03")
print(results_df.to_string())


사용한 입력 컬럼 수: 98
양성 클래스 기준: is_repurchase == 0
precision, recall, f1 는 test 기준
train_roc_auc, test_roc_auc 는 roc_auc_score 기준
과적합 판단 기준: train_roc_auc - test_roc_auc >= 0.03
                    precision  recall      f1  train_roc_auc  test_roc_auc overfit
model                                                                             
CatBoost               0.6443  0.8253  0.7237         0.9259        0.8939  과적합 의심
LightGBM               0.6576  0.7982  0.7211         0.9649        0.8932  과적합 의심
XGBoost                0.6389  0.8261  0.7205         0.9275        0.8930  과적합 의심
SVM                    0.5985  0.7914  0.6816         0.9311        0.8685  과적합 의심
LogisticRegression     0.5828  0.7899  0.6707         0.8643        0.8605  과적합 아님
RandomForest           0.7178  0.6167  0.6634         0.9972        0.8725  과적합 의심
GradientBoosting       0.7387  0.5663  0.6411         0.8982        0.8832  과적합 아님


# 층화 교차 검증 + 3단계 나누기 

In [10]:
import warnings
import pandas as pd

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# 실행 경고 정리부
warnings.filterwarnings(
    "ignore",
    message=(
        "X does not have valid feature names, but "
        "LGBMClassifier was fitted with feature names"
    ),
)

# 출력 형식 정리부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.expand_frame_repr", False)

# 파일 경로 설정부
file_path = (
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable"
    r"\260519_derived_membership_age_specific.csv"
)

# 원본 전체 컬럼 설정부
raw_source_cols = [
    "USER_KEY",
    "product_code",
    "price",
    "billing_method",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "reg_date",
    "reg_hour",
    "end_date",
    "is_repurchase",
]

# 원본 사용 컬럼 설정부
base_feature_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 타깃 컬럼 설정부
target_col = "is_repurchase"

# 실험 설정부
random_state = 42
test_size = 0.2
valid_size_within_non_test = 0.25
cv_splits = 5
overfit_threshold = 0.03

# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)

# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

# 점수 계산 함수부
def evaluate_scores(y_true, y_pred, y_proba):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }

# 과적합 판단 함수부
def judge_overfit(train_roc_auc, eval_roc_auc):
    gap = train_roc_auc - eval_roc_auc
    flag = "과적합 의심" if gap >= overfit_threshold else "과적합 아님"
    return gap, flag

# 입력 컬럼 추출 함수부
def get_feature_columns(df):
    missing_cols = [
        col for col in base_feature_cols + [target_col]
        if col not in df.columns
    ]
    if missing_cols:
        raise ValueError(f"필수 컬럼 누락: {missing_cols}")

    derived_feature_cols = [
        col for col in df.columns
        if col not in raw_source_cols
    ]

    feature_cols = base_feature_cols + derived_feature_cols

    return feature_cols

# 숫자형, 범주형 컬럼 분리 함수부
def split_feature_types(X):
    numeric_features = X.select_dtypes(include="number").columns.tolist()
    categorical_features = [
        col for col in X.columns
        if col not in numeric_features
    ]
    return numeric_features, categorical_features

# 양성 클래스 가중치 계산 함수부
def calculate_scale_pos_weight(y):
    positive_count = int(y.sum())
    negative_count = int(len(y) - positive_count)
    return negative_count / positive_count

# 전처리 파이프라인 생성 함수부
def build_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    return preprocessor

# 모델 생성 함수부
def build_models(y_reference):
    scale_pos_weight = calculate_scale_pos_weight(y_reference)

    models = {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=random_state,
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        ),
        "GradientBoosting": GradientBoostingClassifier(
            random_state=random_state,
        ),
        "SVM": SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale",
            probability=True,
            class_weight="balanced",
            random_state=random_state,
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1,
            scale_pos_weight=scale_pos_weight,
        ),
        "LightGBM": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        ),
        "CatBoost": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            loss_function="Logloss",
            eval_metric="AUC",
            class_weights=[1.0, scale_pos_weight],
            random_seed=random_state,
            verbose=0,
            allow_writing_files=False,
            thread_count=-1,
        ),
    }

    return models

# 파이프라인 생성 함수부
def build_pipeline(model, numeric_features, categorical_features):
    preprocessor = build_preprocessor(
        numeric_features=numeric_features,
        categorical_features=categorical_features,
    )

    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    return clf

# 교차검증 수행 함수부
def run_cross_validation(
    X_train,
    y_train,
    numeric_features,
    categorical_features,
):
    skf = StratifiedKFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=random_state,
    )

    fold_results = []

    for fold_idx, (sub_train_idx, sub_valid_idx) in enumerate(
        skf.split(X_train, y_train),
        start=1,
    ):
        X_sub_train = X_train.iloc[sub_train_idx].copy()
        X_sub_valid = X_train.iloc[sub_valid_idx].copy()
        y_sub_train = y_train.iloc[sub_train_idx].copy()
        y_sub_valid = y_train.iloc[sub_valid_idx].copy()

        models = build_models(y_sub_train)

        for model_name, model in models.items():
            clf = build_pipeline(
                model=model,
                numeric_features=numeric_features,
                categorical_features=categorical_features,
            )

            clf.fit(X_sub_train, y_sub_train)

            y_sub_valid_pred = clf.predict(X_sub_valid)
            y_sub_valid_proba = clf.predict_proba(X_sub_valid)[:, 1]

            sub_valid_scores = evaluate_scores(
                y_true=y_sub_valid,
                y_pred=y_sub_valid_pred,
                y_proba=y_sub_valid_proba,
            )

            fold_results.append(
                {
                    "fold": fold_idx,
                    "model": model_name,
                    "precision": sub_valid_scores["precision"],
                    "recall": sub_valid_scores["recall"],
                    "f1": sub_valid_scores["f1"],
                    "roc_auc": sub_valid_scores["roc_auc"],
                }
            )

    cv_fold_results_df = pd.DataFrame(fold_results)

    cv_summary_df = (
        cv_fold_results_df.groupby("model", as_index=True)
        .agg(
            cv_precision_mean=("precision", "mean"),
            cv_recall_mean=("recall", "mean"),
            cv_f1_mean=("f1", "mean"),
            cv_roc_auc_mean=("roc_auc", "mean"),
        )
        .round(4)
        .sort_values(["cv_f1_mean", "cv_roc_auc_mean"], ascending=False)
    )

    return cv_summary_df

# holdout 평가 수행 함수부
def evaluate_holdout_sets(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    numeric_features,
    categorical_features,
):
    models = build_models(y_train)
    results = []

    for model_name, model in models.items():
        clf = build_pipeline(
            model=model,
            numeric_features=numeric_features,
            categorical_features=categorical_features,
        )

        clf.fit(X_train, y_train)

        y_train_pred = clf.predict(X_train)
        y_train_proba = clf.predict_proba(X_train)[:, 1]

        y_valid_pred = clf.predict(X_valid)
        y_valid_proba = clf.predict_proba(X_valid)[:, 1]

        y_test_pred = clf.predict(X_test)
        y_test_proba = clf.predict_proba(X_test)[:, 1]

        train_scores = evaluate_scores(y_train, y_train_pred, y_train_proba)
        valid_scores = evaluate_scores(y_valid, y_valid_pred, y_valid_proba)
        test_scores = evaluate_scores(y_test, y_test_pred, y_test_proba)

        valid_gap, valid_overfit = judge_overfit(
            train_roc_auc=train_scores["roc_auc"],
            eval_roc_auc=valid_scores["roc_auc"],
        )
        test_gap, test_overfit = judge_overfit(
            train_roc_auc=train_scores["roc_auc"],
            eval_roc_auc=test_scores["roc_auc"],
        )

        results.append(
            {
                "model": model_name,
                "train_roc_auc": train_scores["roc_auc"],
                "valid_precision": valid_scores["precision"],
                "valid_recall": valid_scores["recall"],
                "valid_f1": valid_scores["f1"],
                "valid_roc_auc": valid_scores["roc_auc"],
                "valid_gap": valid_gap,
                "valid_overfit": valid_overfit,
                "test_precision": test_scores["precision"],
                "test_recall": test_scores["recall"],
                "test_f1": test_scores["f1"],
                "test_roc_auc": test_scores["roc_auc"],
                "test_gap": test_gap,
                "test_overfit": test_overfit,
            }
        )

    results_df = pd.DataFrame(results).set_index("model").round(4)

    return results_df

# 분할 요약 생성 함수부
def build_split_summary_df(y_train, y_valid, y_test):
    split_summary_df = pd.DataFrame(
        [
            {
                "split": "train",
                "rows": len(y_train),
                "positive_ratio": round(float(y_train.mean()), 4),
            },
            {
                "split": "valid",
                "rows": len(y_valid),
                "positive_ratio": round(float(y_valid.mean()), 4),
            },
            {
                "split": "test",
                "rows": len(y_test),
                "positive_ratio": round(float(y_test.mean()), 4),
            },
        ]
    )

    return split_summary_df

# 결과 테이블 생성 함수부
def build_result_tables(cv_summary_df, holdout_results_df):
    valid_results_df = (
        holdout_results_df.loc[
            :,
            [
                "train_roc_auc",
                "valid_precision",
                "valid_recall",
                "valid_f1",
                "valid_roc_auc",
                "valid_gap",
                "valid_overfit",
            ],
        ]
        .sort_values(["valid_f1", "valid_roc_auc"], ascending=False)
    )

    model_order = valid_results_df.index.tolist()

    cv_summary_df = cv_summary_df.loc[model_order]

    test_results_df = holdout_results_df.loc[
        model_order,
        [
            "train_roc_auc",
            "test_precision",
            "test_recall",
            "test_f1",
            "test_roc_auc",
            "test_gap",
            "test_overfit",
        ],
    ]

    return cv_summary_df, valid_results_df, test_results_df

# 비과적합 모델 요약 생성 함수부
def build_non_overfit_summary_line(holdout_results_df):
    non_overfit_df = holdout_results_df[
        (holdout_results_df["valid_overfit"] == "과적합 아님")
        & (holdout_results_df["test_overfit"] == "과적합 아님")
    ].copy()

    if non_overfit_df.empty:
        return "과적합 아님 모델 요약: 없음"

    non_overfit_df = non_overfit_df.sort_values(
        ["test_f1", "test_roc_auc"],
        ascending=False,
    )

    summary_parts = []

    for model_name, row in non_overfit_df.iterrows():
        summary_parts.append(
            (
                f"{model_name}"
                f"(valid_f1={row['valid_f1']:.4f}, "
                f"valid_roc_auc={row['valid_roc_auc']:.4f}, "
                f"test_f1={row['test_f1']:.4f}, "
                f"test_roc_auc={row['test_roc_auc']:.4f})"
            )
        )

    return "과적합 아님 모델 요약: " + ", ".join(summary_parts)

# 데이터 로드부
df = pd.read_csv(file_path).copy()

# 사용 컬럼 추출부
feature_cols = get_feature_columns(df)

# 숫자형 변환부
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df[target_col])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 정의부
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
numeric_features, categorical_features = split_feature_types(X)

# train/test 1차 분할 수행부
X_non_test, X_test, y_non_test, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=random_state,
    stratify=y,
)

# train/valid 2차 분할 수행부
X_train, X_valid, y_train, y_valid = train_test_split(
    X_non_test,
    y_non_test,
    test_size=valid_size_within_non_test,
    random_state=random_state,
    stratify=y_non_test,
)

# 분할 요약 생성부
split_summary_df = build_split_summary_df(
    y_train=y_train,
    y_valid=y_valid,
    y_test=y_test,
)

# 교차검증 수행부
cv_summary_df = run_cross_validation(
    X_train=X_train,
    y_train=y_train,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
)

# valid/test holdout 평가 수행부
holdout_results_df = evaluate_holdout_sets(
    X_train=X_train,
    X_valid=X_valid,
    X_test=X_test,
    y_train=y_train,
    y_valid=y_valid,
    y_test=y_test,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
)

# 결과 테이블 생성부
cv_summary_df, valid_results_df, test_results_df = build_result_tables(
    cv_summary_df=cv_summary_df,
    holdout_results_df=holdout_results_df,
)

# 비과적합 모델 요약 생성부
non_overfit_summary_line = build_non_overfit_summary_line(holdout_results_df)

# 결과 출력부
print(f"사용한 입력 컬럼 수: {len(feature_cols)}")
print("양성 클래스 기준: is_repurchase == 0")
print("분할 구조: train 60%, valid 20%, test 20%")
print(f"교차검증 방식: train 데이터 기준 StratifiedKFold {cv_splits}분할")
print(f"과적합 판단 기준: train_roc_auc - eval_roc_auc >= {overfit_threshold}")
print()

print("데이터 분할 요약")
print(split_summary_df.to_string(index=False))
print()

print("교차검증 요약 결과")
print("cv_* 는 train 데이터 5-fold 교차검증 평균 기준")
print(cv_summary_df.to_string())
print()

print("validation 결과")
print("train_roc_auc 는 train 기준")
print("valid_* 는 train으로 학습 후 valid 평가 기준")
print(valid_results_df.to_string())
print()

print("test 결과")
print("train_roc_auc 는 train 기준")
print("test_* 는 같은 train으로 학습 후 처음 보는 test 평가 기준")
print(test_results_df.to_string())
print()

사용한 입력 컬럼 수: 98
양성 클래스 기준: is_repurchase == 0
분할 구조: train 60%, valid 20%, test 20%
교차검증 방식: train 데이터 기준 StratifiedKFold 5분할
과적합 판단 기준: train_roc_auc - eval_roc_auc >= 0.03

데이터 분할 요약
split  rows  positive_ratio
train 14005          0.2845
valid  4669          0.2844
 test  4669          0.2844

교차검증 요약 결과
cv_* 는 train 데이터 5-fold 교차검증 평균 기준
                    cv_precision_mean  cv_recall_mean  cv_f1_mean  cv_roc_auc_mean
model                                                                             
CatBoost                       0.6310          0.8038      0.7069           0.8874
LightGBM                       0.6398          0.7586      0.6940           0.8821
XGBoost                        0.6209          0.7977      0.6982           0.8851
SVM                            0.5968          0.7832      0.6773           0.8639
LogisticRegression             0.5856          0.7769      0.6678           0.8614
RandomForest                   0.7011          0.5870      0.6387          

# 하이퍼파라미터 튜닝

## 1. CatBoost

In [17]:
import time
import warnings

import optuna
import pandas as pd

from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# 실행 경고 정리부
warnings.filterwarnings("ignore")

# Optuna 로그 정리부
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 출력 형식 정리부
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.expand_frame_repr", False)

# 파일 경로 설정부
file_path = (
    r"C:\myCode\ott-churn-prediction\kim.kwangil\derived_variable"
    r"\260519_derived_membership_age_specific.csv"
)

# 원본 전체 컬럼 설정부
raw_source_cols = [
    "USER_KEY",
    "product_code",
    "price",
    "billing_method",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
    "reg_date",
    "reg_hour",
    "end_date",
    "is_repurchase",
]

# 원본 사용 컬럼 설정부
base_feature_cols = [
    "price",
    "max_screen",
    "is_promotion",
    "is_churn_prevented",
    "payment_device",
    "is_user_verified",
    "gender",
    "age",
]

# 타깃 컬럼 설정부
target_col = "is_repurchase"

# 실험 설정부
random_state = 42
test_size = 0.2
valid_size_within_non_test = 0.25
cv_splits = 5
overfit_threshold = 0.03
n_trials = 30

# 이진 변수 변환 함수부
def to_binary(series):
    lowered = series.astype(str).str.strip().str.lower()

    mapped = lowered.map(
        {
            "1": 1,
            "0": 0,
            "y": 1,
            "n": 0,
            "yes": 1,
            "no": 0,
            "true": 1,
            "false": 0,
        }
    )

    numeric = pd.to_numeric(series, errors="coerce")

    return mapped.where(mapped.notna(), numeric)

# OneHotEncoder 버전 호환 함수부
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

# 점수 계산 함수부
def evaluate_scores(y_true, y_pred, y_proba):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_proba),
    }

# 과적합 판단 함수부
def judge_overfit(train_roc_auc, eval_roc_auc):
    gap = train_roc_auc - eval_roc_auc
    flag = "과적합 의심" if gap >= overfit_threshold else "과적합 아님"
    return gap, flag

# 입력 컬럼 추출 함수부
def get_feature_columns(df):
    missing_cols = [
        col for col in base_feature_cols + [target_col]
        if col not in df.columns
    ]
    if missing_cols:
        raise ValueError(f"필수 컬럼 누락: {missing_cols}")

    derived_feature_cols = [
        col for col in df.columns
        if col not in raw_source_cols
    ]

    feature_cols = base_feature_cols + derived_feature_cols

    return feature_cols

# 숫자형, 범주형 컬럼 분리 함수부
def split_feature_types(X):
    numeric_features = X.select_dtypes(include="number").columns.tolist()
    categorical_features = [
        col for col in X.columns
        if col not in numeric_features
    ]
    return numeric_features, categorical_features

# 양성 클래스 가중치 계산 함수부
def calculate_scale_pos_weight(y):
    positive_count = int(y.sum())
    negative_count = int(len(y) - positive_count)
    return negative_count / positive_count

# 전처리 파이프라인 생성 함수부
def build_preprocessor(numeric_features, categorical_features):
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_onehot_encoder()),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ]
    )

    return preprocessor

# CatBoost 모델 생성 함수부
def build_catboost_model(y_reference, params):
    scale_pos_weight = calculate_scale_pos_weight(y_reference)

    model = CatBoostClassifier(
        iterations=params["iterations"],
        learning_rate=params["learning_rate"],
        depth=params["depth"],
        l2_leaf_reg=params["l2_leaf_reg"],
        loss_function="Logloss",
        eval_metric="AUC",
        class_weights=[1.0, scale_pos_weight],
        random_seed=random_state,
        verbose=0,
        allow_writing_files=False,
        thread_count=-1,
    )

    return model

# 파이프라인 생성 함수부
def build_pipeline(model, numeric_features, categorical_features):
    preprocessor = build_preprocessor(
        numeric_features=numeric_features,
        categorical_features=categorical_features,
    )

    clf = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    return clf

# 분할 요약 생성 함수부
def build_split_summary_df(y_train, y_valid, y_test):
    split_summary_df = pd.DataFrame(
        [
            {
                "split": "train",
                "rows": len(y_train),
                "positive_ratio": round(float(y_train.mean()), 4),
            },
            {
                "split": "valid",
                "rows": len(y_valid),
                "positive_ratio": round(float(y_valid.mean()), 4),
            },
            {
                "split": "test",
                "rows": len(y_test),
                "positive_ratio": round(float(y_test.mean()), 4),
            },
        ]
    )

    return split_summary_df

# Trial 진행 출력 콜백 함수부
def build_trial_callback(total_trials):
    def callback(study, trial):
        if trial.state != optuna.trial.TrialState.COMPLETE:
            return

        print(
            f"[trial {trial.number + 1:02d}/{total_trials}] 완료 | "
            f"elapsed_sec={trial.user_attrs['trial_elapsed_sec']:.1f}"
        )
        print(
            f"        CV    : roc_auc_mean={trial.user_attrs['cv_roc_auc_mean']:.4f}, "
            f"f1_mean={trial.user_attrs['cv_f1_mean']:.4f}, "
            f"precision_mean={trial.user_attrs['cv_precision_mean']:.4f}, "
            f"recall_mean={trial.user_attrs['cv_recall_mean']:.4f}"
        )
        print(
            f"        VALID : roc_auc={trial.user_attrs['valid_roc_auc']:.4f}, "
            f"f1={trial.user_attrs['valid_f1']:.4f}, "
            f"precision={trial.user_attrs['valid_precision']:.4f}, "
            f"recall={trial.user_attrs['valid_recall']:.4f}, "
            f"gap={trial.user_attrs['valid_gap']:.4f}, "
            f"overfit={trial.user_attrs['valid_overfit']}"
        )
        print()

    return callback

# Optuna 튜닝 수행 함수부
def run_optuna_tuning(
    X_train,
    X_valid,
    y_train,
    y_valid,
    numeric_features,
    categorical_features,
):
    skf = StratifiedKFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=random_state,
    )

    def objective(trial):
        trial_start_time = time.perf_counter()

        params = {
            "iterations": trial.suggest_int("iterations", 200, 800, step=100),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "depth": trial.suggest_int("depth", 4, 10, step=2),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        }

        cv_rows = []

        for sub_train_idx, sub_valid_idx in skf.split(X_train, y_train):
            X_sub_train = X_train.iloc[sub_train_idx].copy()
            X_sub_valid = X_train.iloc[sub_valid_idx].copy()
            y_sub_train = y_train.iloc[sub_train_idx].copy()
            y_sub_valid = y_train.iloc[sub_valid_idx].copy()

            model = build_catboost_model(
                y_reference=y_sub_train,
                params=params,
            )

            clf = build_pipeline(
                model=model,
                numeric_features=numeric_features,
                categorical_features=categorical_features,
            )

            clf.fit(X_sub_train, y_sub_train)

            y_sub_valid_pred = clf.predict(X_sub_valid)
            y_sub_valid_proba = clf.predict_proba(X_sub_valid)[:, 1]

            cv_scores = evaluate_scores(
                y_true=y_sub_valid,
                y_pred=y_sub_valid_pred,
                y_proba=y_sub_valid_proba,
            )

            cv_rows.append(cv_scores)

        cv_scores_df = pd.DataFrame(cv_rows)

        model = build_catboost_model(
            y_reference=y_train,
            params=params,
        )

        clf = build_pipeline(
            model=model,
            numeric_features=numeric_features,
            categorical_features=categorical_features,
        )

        clf.fit(X_train, y_train)

        y_train_pred = clf.predict(X_train)
        y_train_proba = clf.predict_proba(X_train)[:, 1]

        y_valid_pred = clf.predict(X_valid)
        y_valid_proba = clf.predict_proba(X_valid)[:, 1]

        train_scores = evaluate_scores(y_train, y_train_pred, y_train_proba)
        valid_scores = evaluate_scores(y_valid, y_valid_pred, y_valid_proba)

        valid_gap, valid_overfit = judge_overfit(
            train_roc_auc=train_scores["roc_auc"],
            eval_roc_auc=valid_scores["roc_auc"],
        )

        trial_elapsed_sec = time.perf_counter() - trial_start_time

        trial.set_user_attr("cv_precision_mean", float(cv_scores_df["precision"].mean()))
        trial.set_user_attr("cv_recall_mean", float(cv_scores_df["recall"].mean()))
        trial.set_user_attr("cv_f1_mean", float(cv_scores_df["f1"].mean()))
        trial.set_user_attr("cv_roc_auc_mean", float(cv_scores_df["roc_auc"].mean()))

        trial.set_user_attr("train_roc_auc", float(train_scores["roc_auc"]))

        trial.set_user_attr("valid_precision", float(valid_scores["precision"]))
        trial.set_user_attr("valid_recall", float(valid_scores["recall"]))
        trial.set_user_attr("valid_f1", float(valid_scores["f1"]))
        trial.set_user_attr("valid_roc_auc", float(valid_scores["roc_auc"]))
        trial.set_user_attr("valid_gap", float(valid_gap))
        trial.set_user_attr("valid_overfit", valid_overfit)

        trial.set_user_attr("trial_elapsed_sec", float(trial_elapsed_sec))

        return float(valid_scores["roc_auc"])

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=random_state),
    )

    overall_start_time = time.perf_counter()

    print(
        f"[진행] Optuna CatBoost 튜닝 시작 | "
        f"n_trials={n_trials}, "
        f"목표 지표=valid_roc_auc, "
        f"train 내부 {cv_splits}-fold CV 사용"
    )
    print()

    study.optimize(
        objective,
        n_trials=n_trials,
        callbacks=[build_trial_callback(n_trials)],
        show_progress_bar=False,
    )

    total_elapsed_sec = time.perf_counter() - overall_start_time

    print(f"[진행] Optuna 튜닝 종료 | 총 소요 시간={total_elapsed_sec:.1f}초")
    print()

    return study

# Trial 결과 테이블 생성 함수부
def build_trial_results_df(study):
    trial_rows = []

    for trial in study.trials:
        if trial.state != optuna.trial.TrialState.COMPLETE:
            continue

        trial_rows.append(
            {
                "trial_number": trial.number,
                "iterations": int(trial.params["iterations"]),
                "learning_rate": float(trial.params["learning_rate"]),
                "depth": int(trial.params["depth"]),
                "l2_leaf_reg": float(trial.params["l2_leaf_reg"]),
                "cv_precision_mean": trial.user_attrs["cv_precision_mean"],
                "cv_recall_mean": trial.user_attrs["cv_recall_mean"],
                "cv_f1_mean": trial.user_attrs["cv_f1_mean"],
                "cv_roc_auc_mean": trial.user_attrs["cv_roc_auc_mean"],
                "train_roc_auc": trial.user_attrs["train_roc_auc"],
                "valid_precision": trial.user_attrs["valid_precision"],
                "valid_recall": trial.user_attrs["valid_recall"],
                "valid_f1": trial.user_attrs["valid_f1"],
                "valid_roc_auc": trial.user_attrs["valid_roc_auc"],
                "valid_gap": trial.user_attrs["valid_gap"],
                "valid_overfit": trial.user_attrs["valid_overfit"],
                "trial_elapsed_sec": trial.user_attrs["trial_elapsed_sec"],
            }
        )

    trial_results_df = (
        pd.DataFrame(trial_rows)
        .round(4)
        .sort_values(
            [
                "valid_roc_auc",
                "cv_roc_auc_mean",
                "valid_f1",
                "cv_f1_mean",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    return trial_results_df

# 최적 파라미터 추출 함수부
def build_best_params(trial_results_df):
    best_params = {
        "iterations": int(trial_results_df.loc[0, "iterations"]),
        "learning_rate": float(trial_results_df.loc[0, "learning_rate"]),
        "depth": int(trial_results_df.loc[0, "depth"]),
        "l2_leaf_reg": float(trial_results_df.loc[0, "l2_leaf_reg"]),
    }

    return best_params

# 최종 test 평가 함수부
def evaluate_best_params_on_test(
    best_params,
    trial_results_df,
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    numeric_features,
    categorical_features,
):
    X_train_valid = pd.concat([X_train, X_valid], axis=0)
    y_train_valid = pd.concat([y_train, y_valid], axis=0)

    model = build_catboost_model(
        y_reference=y_train_valid,
        params=best_params,
    )

    clf = build_pipeline(
        model=model,
        numeric_features=numeric_features,
        categorical_features=categorical_features,
    )

    clf.fit(X_train_valid, y_train_valid)

    y_train_valid_pred = clf.predict(X_train_valid)
    y_train_valid_proba = clf.predict_proba(X_train_valid)[:, 1]

    y_test_pred = clf.predict(X_test)
    y_test_proba = clf.predict_proba(X_test)[:, 1]

    train_valid_scores = evaluate_scores(
        y_true=y_train_valid,
        y_pred=y_train_valid_pred,
        y_proba=y_train_valid_proba,
    )

    test_scores = evaluate_scores(
        y_true=y_test,
        y_pred=y_test_pred,
        y_proba=y_test_proba,
    )

    test_gap, test_overfit = judge_overfit(
        train_roc_auc=train_valid_scores["roc_auc"],
        eval_roc_auc=test_scores["roc_auc"],
    )

    best_summary_df = pd.DataFrame(
        [
            {
                "stage": "cv_mean",
                "precision": trial_results_df.loc[0, "cv_precision_mean"],
                "recall": trial_results_df.loc[0, "cv_recall_mean"],
                "f1": trial_results_df.loc[0, "cv_f1_mean"],
                "roc_auc": trial_results_df.loc[0, "cv_roc_auc_mean"],
                "gap": None,
                "overfit": None,
            },
            {
                "stage": "valid",
                "precision": trial_results_df.loc[0, "valid_precision"],
                "recall": trial_results_df.loc[0, "valid_recall"],
                "f1": trial_results_df.loc[0, "valid_f1"],
                "roc_auc": trial_results_df.loc[0, "valid_roc_auc"],
                "gap": trial_results_df.loc[0, "valid_gap"],
                "overfit": trial_results_df.loc[0, "valid_overfit"],
            },
            {
                "stage": "test",
                "precision": round(float(test_scores["precision"]), 4),
                "recall": round(float(test_scores["recall"]), 4),
                "f1": round(float(test_scores["f1"]), 4),
                "roc_auc": round(float(test_scores["roc_auc"]), 4),
                "gap": round(float(test_gap), 4),
                "overfit": test_overfit,
            },
        ]
    )

    final_test_result_df = pd.DataFrame(
        [
            {
                "iterations": best_params["iterations"],
                "learning_rate": best_params["learning_rate"],
                "depth": best_params["depth"],
                "l2_leaf_reg": best_params["l2_leaf_reg"],
                "train_valid_roc_auc": round(float(train_valid_scores["roc_auc"]), 4),
                "test_precision": round(float(test_scores["precision"]), 4),
                "test_recall": round(float(test_scores["recall"]), 4),
                "test_f1": round(float(test_scores["f1"]), 4),
                "test_roc_auc": round(float(test_scores["roc_auc"]), 4),
                "test_gap": round(float(test_gap), 4),
                "test_overfit": test_overfit,
            }
        ]
    )

    return best_summary_df, final_test_result_df

# 데이터 로드부
print("[진행] 데이터 로드 시작")
df = pd.read_csv(file_path).copy()

# 사용 컬럼 추출부
feature_cols = get_feature_columns(df)

# 숫자형 변환부
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["max_screen"] = pd.to_numeric(df["max_screen"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce")

# 이진형 변환부
binary_cols = [
    "is_promotion",
    "is_churn_prevented",
    "is_user_verified",
]

for col in binary_cols:
    df[col] = to_binary(df[col])

df["is_repurchase_num"] = to_binary(df[target_col])

# 타깃 결측 제거부
df = df[df["is_repurchase_num"].isin([0, 1])].copy()

# 입력 변수, 타깃 변수 생성부
X = df[feature_cols].copy()

# 양성 클래스 정의부
# is_repurchase == 0 예측 목표 정의부
y = (df["is_repurchase_num"] == 0).astype(int)

# 숫자형, 범주형 컬럼 구분부
numeric_features, categorical_features = split_feature_types(X)

print(
    f"[진행] 데이터 준비 완료 | rows={len(df)}, "
    f"feature_count={len(feature_cols)}, "
    f"numeric_features={len(numeric_features)}, "
    f"categorical_features={len(categorical_features)}"
)

# train/test 1차 분할 수행부
print("[진행] train/test 1차 분할 수행")
X_non_test, X_test, y_non_test, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=random_state,
    stratify=y,
)

# train/valid 2차 분할 수행부
print("[진행] train/valid 2차 분할 수행")
X_train, X_valid, y_train, y_valid = train_test_split(
    X_non_test,
    y_non_test,
    test_size=valid_size_within_non_test,
    random_state=random_state,
    stratify=y_non_test,
)

print(
    f"[진행] 분할 완료 | train={len(X_train)}, "
    f"valid={len(X_valid)}, test={len(X_test)}"
)
print()

# 분할 요약 생성부
split_summary_df = build_split_summary_df(
    y_train=y_train,
    y_valid=y_valid,
    y_test=y_test,
)

# Optuna 튜닝 수행부
study = run_optuna_tuning(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
)

# Trial 결과 테이블 생성부
trial_results_df = build_trial_results_df(study=study)

# 최적 파라미터 추출부
best_params = build_best_params(trial_results_df=trial_results_df)

# 최종 test 평가 수행부
best_summary_df, final_test_result_df = evaluate_best_params_on_test(
    best_params=best_params,
    trial_results_df=trial_results_df,
    X_train=X_train,
    X_valid=X_valid,
    X_test=X_test,
    y_train=y_train,
    y_valid=y_valid,
    y_test=y_test,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
)

# 결과 출력부
print("=" * 100)
print(f"사용한 입력 컬럼 수: {len(feature_cols)}")
print("양성 클래스 기준: is_repurchase == 0")
print("분할 구조: train 60%, valid 20%, test 20%")
print(f"교차검증 방식: train 데이터 기준 StratifiedKFold {cv_splits}분할")
print(f"Optuna trial 수: {n_trials}")
print("튜닝 파라미터: iterations, learning_rate, depth, l2_leaf_reg")
print("최고 trial 선정 기준: valid_roc_auc -> cv_roc_auc_mean -> valid_f1 -> cv_f1_mean")
print(f"과적합 판단 기준: train_roc_auc - eval_roc_auc >= {overfit_threshold}")
print()

print("데이터 분할 요약")
print(split_summary_df.to_string(index=False))
print()

print("Optuna trial 결과")
print("cv_* 는 train 데이터 5-fold 교차검증 평균 기준")
print("valid_* 는 train으로 학습 후 valid 평가 기준")
print(trial_results_df.to_string(index=False))
print()

print("최종 최고 파라미터 조합")
print(best_params)
print()

print("최고 조합 성능 요약")
print(best_summary_df.to_string(index=False))
print()

print("최고 조합 기준 최종 test 결과")
print("train_valid_roc_auc 는 train+valid 재학습 기준")
print("test_* 는 처음 보는 test 평가 기준")
print(final_test_result_df.to_string(index=False))
print("=" * 100)

[진행] 데이터 로드 시작
[진행] 데이터 준비 완료 | rows=23343, feature_count=98, numeric_features=96, categorical_features=2
[진행] train/test 1차 분할 수행
[진행] train/valid 2차 분할 수행
[진행] 분할 완료 | train=14005, valid=4669, test=4669

[진행] Optuna CatBoost 튜닝 시작 | n_trials=30, 목표 지표=valid_roc_auc, train 내부 5-fold CV 사용

[trial 01/30] 완료 | elapsed_sec=43.2
        CV    : roc_auc_mean=0.8811, f1_mean=0.6936, precision_mean=0.6626, recall_mean=0.7277
        VALID : roc_auc=0.8853, f1=0.7017, precision=0.6727, recall=0.7334, gap=0.1083, overfit=과적합 의심

[trial 02/30] 완료 | elapsed_sec=14.8
        CV    : roc_auc_mean=0.8652, f1_mean=0.6714, precision_mean=0.5761, recall_mean=0.8048
        VALID : roc_auc=0.8632, f1=0.6675, precision=0.5792, recall=0.7877, gap=0.0098, overfit=과적합 아님

[trial 03/30] 완료 | elapsed_sec=27.4
        CV    : roc_auc_mean=0.8886, f1_mean=0.7082, precision_mean=0.6308, recall_mean=0.8073
        VALID : roc_auc=0.8888, f1=0.7075, precision=0.6344, recall=0.7997, gap=0.0388, overfit=과적합 의심

[tr

[W 2026-05-18 21:05:58,476] Trial 20 failed with parameters: {'iterations': 500, 'learning_rate': 0.06992451049005777, 'depth': 4, 'l2_leaf_reg': 5.736330003702177} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "c:\Users\user\anaconda3\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\user\AppData\Local\Temp\ipykernel_9052\4012496692.py", line 345, in objective
    clf.fit(X_train, y_train)
    ~~~~~~~^^^^^^^^^^^^^^^^^^
  File "c:\Users\user\anaconda3\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\user\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\user\anaconda3\Lib\site-packages\catboost\core.py", line 5547, in fit
    self._

KeyboardInterrupt: 